<a href="https://colab.research.google.com/github/pandahsu849/Programing_Languages/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-generativeai

In [ ]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
from google.colab import userdata

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
genai.configure(api_key=api_key)

model = genai.GenerativeModel('gemini-2.5-flash')

In [ ]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1CQZDzhSqoIbR8JjExDr9eTq0qDzP6vQvUp9JAs4Jmic/edit?usp=sharing"
WORKSHEET_NAME = "工作表3"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [ ]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [ ]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """

    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = model.generate_content(prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [ ]:
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：88
已記錄：日期: 2026-03-26, 科目: 國文, 成績: 88

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：100
已記錄：日期: 2026-03-26, 科目: 數學, 成績: 100

請輸入科目（或輸入 'q' 停止）：社會
請輸入 社會 的成績：70
已記錄：日期: 2026-03-26, 科目: 社會, 成績: 70

請輸入科目（或輸入 'q' 停止）：自然
請輸入 自然 的成績：100
已記錄：日期: 2026-03-26, 科目: 自然, 成績: 100

請輸入科目（或輸入 'q' 停止）：q


In [ ]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'好的，這是一份根據您提供的學生單次成績列表所做的簡單摘要與常見迷思整理，全程不對學生的表現進行評分或價值判斷。\n\n---\n\n### 學生成績簡單摘要\n\n以下是該學生在**2026年3月26日**的成績報告，涵蓋了四個主要科目：\n\n*   **國文：88分**\n*   **數學：100分**\n*   **社會：70分**\n*   **自然：100分**\n\n整體而言，這次的成績分佈顯示在數學和自然科目中獲得滿分，國文科目取得良好分數，社會科目則落在較低區間。\n\n---\n\n### 關於學生成績的常見迷思整理\n\n在看待學生的成績時，人們常會有一些既定的觀念或誤解。以下整理幾項常見的迷思，旨在提供一個更全面的視角：\n\n1.  **迷思一：成績數字代表一切學習成果**\n    *   **真實情況：** 成績單上的數字是衡量學生在特定時間點對特定課程內容掌握程度的一種方式。然而，它往往無法完全體現學生在學習過程中的努力、批判性思維、解決問題能力、創造力、團隊合作精神，以及對學習的熱情和好奇心。這些軟實力對於學生的長遠發展同樣重要，甚至更為關鍵。\n\n2.  **迷思二：高分科目代表全無問題，低分科目表示能力不足**\n    *   **真實情況：** 學生在不同科目上有不同的表現是正常的。高分可能來自於學生對該科有濃厚興趣、學習方法得當，或該次測驗內容恰好是其強項。而低分則可能有多種原因，例如：該科內容對學生而言較為抽象、學習方法尚未找到最佳匹配、考試時的情緒或壓力、教師的教學風格差異，甚至只是單次測驗的疏忽。這並不必然代表學生在該領域的「能力不足」，而是一個需要進一步探索和理解的訊號。\n\n3.  **迷思三：成績高低等同於智力高低**\n    *   **真實情況：** 智力是一個複雜且多面向的概念，成績高低與智力並非簡單的等號關係。成績更直接反映的是學生在特定教育體系下，對課程知識的理解、記憶、應用能力，以及應試技巧。許多因素如學習習慣、專注力、學習策略、時間管理、抗壓性，甚至是當天的身體狀況，都會影響成績表現，這些都與智力的定義有所區隔。\n\n4.  **迷思四：應將成績與他人比較以評估自身價值**\n    *   **真實情況：** 每個學生的學習進度和優勢領域都是獨特的。過度將成績與他人比較，可能會導致不必要

In [ ]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row + i, 3, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：60
已記錄：日期: 2026-03-26, 科目: 國文, 成績: 60

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：100
已記錄：日期: 2026-03-26, 科目: 數學, 成績: 100

請輸入科目（或輸入 'q' 停止）：社會
請輸入 社會 的成績：60
已記錄：日期: 2026-03-26, 科目: 社會, 成績: 60

請輸入科目（或輸入 'q' 停止）：自然
請輸入 自然 的成績：75
已記錄：日期: 2026-03-26, 科目: 自然, 成績: 75

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，這是一份根據您提供的成績列表所做的簡單摘要與常見迷思整理：

---

### **學生單次成績摘要 (2026年3月26日)**

這份成績清單呈現了學生在2026年3月26日四個科目的單次測驗表現：

*   **國文：** 60分
*   **數學：** 100分
*   **社會：** 60分
*   **自然：** 75分

**總結而言：**

*   **優異表現：** 該學生在**數學科目**表現突出，取得了滿分。
*   **良好表現：** **自然科目**表現中等偏上。
*   **相對較低：** **國文**與**社會科目**的成績則相對較低。

---

### **常見迷思整理（關於單次成績解讀）**

在檢視這類單次成績列表時，我們應避免以下一些常見的迷思與誤解：

1.  **迷思一：單次成績代表學生全部的能力或未來趨勢。**
    *   **釐清：** 這僅是學生在特定時間點、針對特定範圍的表現。它可能受到多種因素影響（例如當天狀態、考題難度、準備程度或考試範圍側重），不能完全反映學生的長期學習狀況、科目整體能力，